# 01 — Análise Exploratória de Dados (EDA)

Antes de sair treinando modelos, precisamos entender os dados. Essa etapa é chamada de EDA (Exploratory Data Analysis) e é uma das mais importantes em qualquer projeto de ML.

Aqui vamos responder perguntas como:
- Como é a distribuição dos preços?
- Quais características mais influenciam o preço de uma casa?
- Tem muitos valores faltando?
- O bairro importa muito?

## 1. Imports e Configurações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Adiciona o diretório raiz ao path para importar os módulos do projeto
sys.path.append(os.path.join(os.getcwd(), '..'))

# Deixa os gráficos mais bonitos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('✅ Tudo importado!')

## 2. Carregar os Dados

In [ ]:
# Carrega o dataset de treino
df = pd.read_csv('../data/train.csv')

print(f'Linhas: {df.shape[0]}')
print(f'Colunas: {df.shape[1]}')
df.head()

## 3. Distribuição do Preço (Variável Alvo)

O preço das casas tem uma distribuição assimétrica — existem poucas casas muito caras que puxam a média para cima. Por isso usamos log-transform no alvo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição original — note o "rabo" para a direita (assimetria)
sns.histplot(df['SalePrice'], kde=True, ax=axes[0], color='#4C72B0')
axes[0].set_title('SalePrice — Distribuição Original')
axes[0].set_xlabel('Preço (USD)')

# Após log-transform — muito mais parecida com uma distribuição normal
sns.histplot(np.log1p(df['SalePrice']), kde=True, ax=axes[1], color='#55A868')
axes[1].set_title('log(SalePrice) — Mais Normal')
axes[1].set_xlabel('log(Preço)')

plt.tight_layout()
plt.show()

print(f'Assimetria original:          {df["SalePrice"].skew():.2f}')
print(f'Assimetria após log-transform: {np.log1p(df["SalePrice"]).skew():.2f}')

## 4. Correlação com o Preço

Quais variáveis numéricas têm maior correlação com o SalePrice?

In [ ]:
# Pega as top 15 features mais correlacionadas com o preço
correlacoes = df.select_dtypes(include=[np.number]).corr()
top_corr = correlacoes['SalePrice'].abs().sort_values(ascending=False).head(15).index

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    df[top_corr].corr(),
    annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=0.5, ax=ax
)
ax.set_title('Correlação — Top 15 Features com SalePrice')
plt.tight_layout()
plt.show()

## 5. Valores Nulos

Muitas colunas têm valores nulos porque o imóvel simplesmente não tem aquela característica (ex: sem piscina, sem garagem). Isso é importante para o pré-processamento.

In [ ]:
# Calcula a porcentagem de nulos por coluna
nulos = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
nulos = nulos[nulos > 0]

print(f'Colunas com valores nulos: {len(nulos)}')

fig, ax = plt.subplots(figsize=(10, 6))
nulos.sort_values().plot(kind='barh', ax=ax, color='#C44E52')
ax.set_title('% de Valores Nulos por Coluna')
ax.set_xlabel('% de Nulos')
plt.tight_layout()
plt.show()

## 6. Features Mais Importantes vs Preço

In [ ]:
# Vamos olhar as features mais correlacionadas individualmente
top_features = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'FullBath']

fig, axes = plt.subplots(1, len(top_features), figsize=(18, 5))

for i, feat in enumerate(top_features):
    if df[feat].nunique() < 12:
        # Boxplot para variáveis discretas (poucos valores únicos)
        sns.boxplot(data=df, x=feat, y='SalePrice', ax=axes[i], palette='Blues')
    else:
        # Scatter para variáveis contínuas
        axes[i].scatter(df[feat], df['SalePrice'], alpha=0.3, s=10, color='#4C72B0')
        axes[i].set_xlabel(feat)
        axes[i].set_ylabel('SalePrice')
    axes[i].set_title(feat)

plt.suptitle('Top Features vs SalePrice', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 7. Preço por Bairro

O bairro é um dos fatores mais importantes no preço de um imóvel!

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
mediana_bairro = df.groupby('Neighborhood')['SalePrice'].median().sort_values()
mediana_bairro.plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_title('Preço Mediano por Bairro')
ax.set_xlabel('Preço Mediano (USD)')
plt.tight_layout()
plt.show()